In [1]:
import pandas as pd

customers_df = pd.DataFrame({
    "customer_id":[1,2,3,4,5,6,7,8],
    "name":["Alice","Bob","Charlie","Diana","Eve","Frank","Grace","Henry"],
    "signup_date":pd.to_datetime([
        "2023-01-15","2022-11-20","2023-03-10","2021-07-05",
        "2022-12-12","2023-02-01","2023-03-25","2022-08-17"
    ]),
    "country":["UK","France","UK","Germany","France","UK","Germany","UK"],
    "loyalty_tier":["Gold","Silver","Gold","Gold","Silver","Bronze","Silver","Gold"]
})

products_df = pd.DataFrame({
    "product_id":[101,102,103,104,105],
    "product_name":["Laptop","Headphones","Coffee Maker","Smartphone","Desk Lamp"],
    "category":["Electronics","Electronics","Home","Electronics","Home"],
    "price":[1200,200,80,800,45]
})

orders_df = pd.DataFrame({
    "order_id":[1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011],
    "customer_id":[1,2,3,1,4,5,6,2,3,7,8],
    "product_id":[101,102,103,104,105,101,102,103,104,105,101],
    "quantity":[1,2,1,1,3,2,1,1,1,1,1],
    "order_date":pd.to_datetime([
        "2024-01-01","2024-01-02","2024-01-02","2024-01-03",
        "2024-01-04","2024-01-04","2024-01-05","2024-01-06",
        "2024-01-07","2024-01-08","2024-01-09"
    ])
})

In [2]:
'''
Task 1 - Build master table
'''
master_df = (
    orders_df
    .merge(customers_df,on='customer_id',how='left')
    .merge(products_df,on='product_id',how='left')
)
master_df

,order_id,customer_id,product_id,quantity,order_date,name,signup_date,country,loyalty_tier,product_name,category,price
0,1001,1,101,1,2024-01-01,Alice,2023-01-15,UK,Gold,Laptop,Electronics,1200
1,1002,2,102,2,2024-01-02,Bob,2022-11-20,France,Silver,Headphones,Electronics,200
2,1003,3,103,1,2024-01-02,Charlie,2023-03-10,UK,Gold,Coffee Maker,Home,80
3,1004,1,104,1,2024-01-03,Alice,2023-01-15,UK,Gold,Smartphone,Electronics,800
4,1005,4,105,3,2024-01-04,Diana,2021-07-05,Germany,Gold,Desk Lamp,Home,45
5,1006,5,101,2,2024-01-04,Eve,2022-12-12,France,Silver,Laptop,Electronics,1200
6,1007,6,102,1,2024-01-05,Frank,2023-02-01,UK,Bronze,Headphones,Electronics,200
7,1008,2,103,1,2024-01-06,Bob,2022-11-20,France,Silver,Coffee Maker,Home,80
8,1009,3,104,1,2024-01-07,Charlie,2023-03-10,UK,Gold,Smartphone,Electronics,800
9,1010,7,105,1,2024-01-08,Grace,2023-03-25,Germany,Silver,Desk Lamp,Home,45


In [3]:
'''
Task 2 - Build customer features
'''
customer_features = (
    master_df
    .assign(order_value = lambda x: x["price"] * x["quantity"])  # feature engineering
    .groupby(["customer_id", "country", "loyalty_tier"], as_index=False)
    .agg(
        total_orders=("order_id", "count"),
        unique_products=("product_id", "nunique"),
        avg_order_value=("order_value", "mean")
    )
)
customer_features

,customer_id,country,loyalty_tier,total_orders,unique_products,avg_order_value
0,1,UK,Gold,2,2,1000.0
1,2,France,Silver,2,2,240.0
2,3,UK,Gold,2,2,440.0
3,4,Germany,Gold,1,1,135.0
4,5,France,Silver,1,1,2400.0
5,6,UK,Bronze,1,1,200.0
6,7,Germany,Silver,1,1,45.0
7,8,UK,Gold,1,1,1200.0


In [4]:
'''
Task 3 - Create target variable
'''
customer_features = customer_features.assign(
    repeat_purchase = lambda x: (x["total_orders"] > 1).astype(int)
)
customer_features

,customer_id,country,loyalty_tier,total_orders,unique_products,avg_order_value,repeat_purchase
0,1,UK,Gold,2,2,1000.0,1
1,2,France,Silver,2,2,240.0,1
2,3,UK,Gold,2,2,440.0,1
3,4,Germany,Gold,1,1,135.0,0
4,5,France,Silver,1,1,2400.0,0
5,6,UK,Bronze,1,1,200.0,0
6,7,Germany,Silver,1,1,45.0,0
7,8,UK,Gold,1,1,1200.0,0
